# Breakfast TDA Descriptor Experiment

Este notebook evalua si los descriptores topologicos (TDA frame-level + curvas) muestran evidencia de segmentacion **a priori**
antes de entrenar un modelo temporal profundo.

## Objetivos

1. Visualizar señales topologicas por video con overlay de boundaries GT.
2. Medir separabilidad de action units usando descriptores topologicos.
3. Evaluar un detector simple de boundaries basado en picos sobre señales topologicas.
4. Exportar resultados graficos y resumen cuantitativo.

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(7)

In [ ]:
# Configuracion base
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pipeline').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'pipeline').exists():
            PROJECT_ROOT = parent
            break

ARTIFACTS_BREAKFAST = PROJECT_ROOT / 'pipeline/pipeline2_tda_dl/artifacts_breakfast'
CUBICAL_MANIFEST = ARTIFACTS_BREAKFAST / 'outputs_cubical/manifest_cubical.json'
CURVES_MANIFEST = ARTIFACTS_BREAKFAST / 'outputs_curves/manifest_curves.json'
FRAME_LABELS_MANIFEST = ARTIFACTS_BREAKFAST / 'frame_labels/manifest_frame_labels.json'
LABEL_MAP_PATH = ARTIFACTS_BREAKFAST / 'frame_labels/label_map.json'

EXPERIMENT_DIR = PROJECT_ROOT / 'pipeline/pipeline2_tda_dl/artifacts_breakfast_experiments'
FIGURES_DIR = EXPERIMENT_DIR / 'figures'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Parametros del experimento
SPLIT = 'test'
MAX_VIDEOS = 8
ACTIVITIES = ['cereals', 'pancakes', 'coffee', 'tea']
BOUNDARY_TOLERANCE_SEC = 0.5
SMOOTH_WINDOW = 5

print('PROJECT_ROOT      :', PROJECT_ROOT)
print('ARTIFACTS_BREAKFAST:', ARTIFACTS_BREAKFAST)
print('EXPERIMENT_DIR    :', EXPERIMENT_DIR)

In [ ]:
@dataclass
class VideoData:
    video_id: str
    split: str
    activity_label: str
    subject_id: str
    timestamps: np.ndarray
    tda_features: np.ndarray
    curve_signals: np.ndarray
    curve_labels: List[str]
    frame_label_ids: np.ndarray


def _read_json_list(path: Path) -> List[dict]:
    if not path.exists():
        return []
    payload = json.loads(path.read_text(encoding='utf-8'))
    if isinstance(payload, list):
        return payload
    return []


def _read_json_dict(path: Path) -> Dict:
    if not path.exists():
        return {}
    payload = json.loads(path.read_text(encoding='utf-8'))
    if isinstance(payload, dict):
        return payload
    return {}


def _resolve_path(raw_path: str, manifest_path: Path) -> Path:
    path = Path(raw_path)
    if path.is_absolute():
        return path
    return (manifest_path.parent / path).resolve()


def _normalize_video_id(value: str) -> str:
    stem = Path(str(value)).stem
    if stem.endswith('_curves'):
        stem = stem[:-7]
    if stem.endswith('_labels'):
        stem = stem[:-7]
    return stem.lower()


def _moving_average_1d(signal: np.ndarray, window: int) -> np.ndarray:
    if window <= 1 or signal.size == 0:
        return signal.astype(np.float32)
    kernel = np.ones((window,), dtype=np.float32) / float(window)
    return np.convolve(signal, kernel, mode='same').astype(np.float32)


def _zscore(x: np.ndarray) -> np.ndarray:
    if x.size == 0:
        return x.astype(np.float32)
    mean = float(np.mean(x))
    std = float(np.std(x))
    if std < 1e-8:
        std = 1.0
    return ((x - mean) / std).astype(np.float32)


def _segment_boundaries_from_labels(label_ids: np.ndarray, timestamps: np.ndarray) -> np.ndarray:
    if label_ids.size <= 1:
        return np.zeros((0,), dtype=np.float32)
    change_idx = np.where(np.diff(label_ids) != 0)[0] + 1
    return timestamps[change_idx].astype(np.float32)


def _find_peaks(signal: np.ndarray, threshold: float) -> np.ndarray:
    if signal.size < 3:
        return np.zeros((0,), dtype=np.int32)
    idxs = []
    for i in range(1, signal.shape[0] - 1):
        if signal[i] >= threshold and signal[i] > signal[i - 1] and signal[i] >= signal[i + 1]:
            idxs.append(i)
    return np.asarray(idxs, dtype=np.int32)


def _match_events(pred_times: np.ndarray, gt_times: np.ndarray, tol: float) -> Tuple[int, int, int, List[float]]:
    if pred_times.size == 0 and gt_times.size == 0:
        return 0, 0, 0, []
    if gt_times.size == 0:
        return 0, int(pred_times.size), 0, []
    if pred_times.size == 0:
        return 0, 0, int(gt_times.size), []

    used = np.zeros((gt_times.shape[0],), dtype=bool)
    tp = 0
    dists: List[float] = []
    for p in pred_times:
        best = -1
        best_dist = float('inf')
        for i, g in enumerate(gt_times):
            if used[i]:
                continue
            d = abs(float(p) - float(g))
            if d <= tol and d < best_dist:
                best = i
                best_dist = d
        if best >= 0:
            used[best] = True
            tp += 1
            dists.append(best_dist)
    fp = int(pred_times.size) - tp
    fn = int(gt_times.size) - tp
    return tp, fp, fn, dists

In [ ]:
# Carga de manifests
cubical_rows = _read_json_list(CUBICAL_MANIFEST)
curves_rows = _read_json_list(CURVES_MANIFEST)
frame_label_rows = _read_json_list(FRAME_LABELS_MANIFEST)
label_map = _read_json_dict(LABEL_MAP_PATH)
id_to_label = {int(v): str(k) for k, v in label_map.items()} if label_map else {}

print('cubical rows:', len(cubical_rows))
print('curves rows :', len(curves_rows))
print('label rows  :', len(frame_label_rows))
print('num labels  :', len(label_map))

if not cubical_rows or not curves_rows or not frame_label_rows:
    raise RuntimeError(
        'Faltan artefactos Breakfast. Ejecuta primero: '        'breakfast_manifest_builder -> breakfast_cubical_preprocessing -> breakfast_curves -> build_frame_labels.'
    )

In [ ]:
# Indexes para alinear video_id -> paths
cubical_index: Dict[str, dict] = {}
for row in cubical_rows:
    vid = str(row.get('video_id') or row.get('source_path') or row.get('output_path') or '').strip()
    out = str(row.get('output_path') or '').strip()
    if not vid or not out:
        continue
    key = _normalize_video_id(vid)
    cubical_index[key] = {
        'video_id': Path(vid).stem,
        'split': str(row.get('split') or ''),
        'activity_label': str(row.get('activity_label') or ''),
        'subject_id': str(row.get('subject_id') or ''),
        'npz_path': _resolve_path(out, CUBICAL_MANIFEST),
    }

curves_index: Dict[str, Path] = {}
for row in curves_rows:
    vid = str(row.get('video_id') or row.get('source_path') or row.get('output_path') or '').strip()
    out = str(row.get('output_path') or '').strip()
    if not vid or not out:
        continue
    key = _normalize_video_id(vid)
    curves_index[key] = _resolve_path(out, CURVES_MANIFEST)

labels_index: Dict[str, dict] = {}
for row in frame_label_rows:
    vid = str(row.get('video_id') or row.get('output_path') or '').strip()
    out = str(row.get('output_path') or '').strip()
    if not vid or not out:
        continue
    key = _normalize_video_id(vid)
    labels_index[key] = {
        'video_id': Path(vid).stem,
        'split': str(row.get('split') or ''),
        'npz_path': _resolve_path(out, FRAME_LABELS_MANIFEST),
    }

available_keys = sorted(set(cubical_index) & set(curves_index) & set(labels_index))
print('aligned videos:', len(available_keys))

In [ ]:
# Seleccion de videos para experimento
selected_keys: List[str] = []
for key in available_keys:
    c = cubical_index[key]
    if SPLIT and c.get('split', '') != SPLIT:
        continue
    activity = c.get('activity_label', '')
    if ACTIVITIES and activity and activity not in ACTIVITIES:
        continue
    selected_keys.append(key)

if len(selected_keys) < MAX_VIDEOS:
    for key in available_keys:
        if key in selected_keys:
            continue
        c = cubical_index[key]
        if SPLIT and c.get('split', '') != SPLIT:
            continue
        selected_keys.append(key)
        if len(selected_keys) >= MAX_VIDEOS:
            break

selected_keys = selected_keys[:MAX_VIDEOS]
print('selected videos:', len(selected_keys))
print([cubical_index[k]['video_id'] for k in selected_keys])

if not selected_keys:
    raise RuntimeError('No hay videos seleccionados. Revisa SPLIT/ACTIVITIES y manifests.')

In [ ]:
# Cargar datos por video
videos: List[VideoData] = []
for key in selected_keys:
    c = cubical_index[key]
    curves_path = curves_index[key]
    labels_path = labels_index[key]['npz_path']

    if not c['npz_path'].exists() or not curves_path.exists() or not labels_path.exists():
        print(f'[skip-missing] {c["video_id"]}')
        continue

    with np.load(c['npz_path']) as data:
        timestamps = data['timestamps_sec'].astype(np.float32)
        tda_features = data['tda_features'].astype(np.float32)

    with np.load(curves_path) as data:
        curve_signals = data['curve_signals'].astype(np.float32)
        curve_labels = [str(x) for x in data['curve_labels'].tolist()]

    with np.load(labels_path) as data:
        frame_label_ids = data['frame_label_ids'].astype(np.int32)

    min_len = min(len(timestamps), tda_features.shape[0], curve_signals.shape[0], frame_label_ids.shape[0])
    if min_len <= 2:
        continue

    videos.append(
        VideoData(
            video_id=c['video_id'],
            split=c['split'],
            activity_label=c['activity_label'],
            subject_id=c['subject_id'],
            timestamps=timestamps[:min_len],
            tda_features=tda_features[:min_len],
            curve_signals=curve_signals[:min_len],
            curve_labels=curve_labels,
            frame_label_ids=frame_label_ids[:min_len],
        )
    )

print('loaded videos:', len(videos))
if not videos:
    raise RuntimeError('No se pudo cargar ningun video del subconjunto seleccionado.')

In [ ]:
# Visualizacion temporal por video

def _curve(video: VideoData, name: str) -> np.ndarray:
    if name in video.curve_labels:
        idx = video.curve_labels.index(name)
        return video.curve_signals[:, idx].astype(np.float32)
    return np.zeros((video.curve_signals.shape[0],), dtype=np.float32)

for video in videos:
    ts = video.timestamps
    gt_boundaries = _segment_boundaries_from_labels(video.frame_label_ids, ts)

    combined = _curve(video, 'combined_activity')
    combined_delta = _curve(video, 'combined_activity_delta')
    if np.allclose(combined_delta, 0.0) and combined.size > 1:
        combined_delta = np.hstack([np.array([0.0], dtype=np.float32), np.abs(np.diff(combined))]).astype(np.float32)

    z_signal = _curve(video, 'combined_activity_z')
    if np.allclose(z_signal, 0.0):
        z_signal = _zscore(combined)

    bright_delta = _curve(video, 'brightness_delta')
    score = _zscore(np.abs(combined_delta)) + 0.35 * _zscore(np.abs(bright_delta))
    score = _moving_average_1d(score, SMOOTH_WINDOW)

    threshold = float(np.quantile(score, 0.9))
    peak_idx = _find_peaks(score, threshold)
    peak_times = ts[peak_idx] if peak_idx.size else np.zeros((0,), dtype=np.float32)

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(f'{video.video_id} | split={video.split} | activity={video.activity_label}', fontsize=12)

    axes[0].plot(ts, combined, lw=1.3, label='combined_activity')
    axes[0].plot(ts, z_signal, lw=1.1, label='combined_activity_z')
    axes[0].legend(loc='upper right')
    axes[0].set_ylabel('activity')

    axes[1].plot(ts, combined_delta, lw=1.2, label='combined_activity_delta')
    axes[1].plot(ts, bright_delta, lw=1.0, label='brightness_delta')
    axes[1].legend(loc='upper right')
    axes[1].set_ylabel('deltas')

    axes[2].plot(ts, score, lw=1.3, color='tab:blue', label='boundary score')
    axes[2].axhline(threshold, color='tab:red', ls='--', lw=1.0, label='q90 threshold')
    if peak_times.size:
        axes[2].scatter(peak_times, score[peak_idx], color='tab:red', s=16, zorder=3, label='pred peaks')
    axes[2].legend(loc='upper right')
    axes[2].set_ylabel('score')

    axes[3].plot(ts, video.frame_label_ids, drawstyle='steps-post', lw=1.2, color='tab:green')
    axes[3].set_ylabel('GT label id')
    axes[3].set_xlabel('time [sec]')

    for ax in axes:
        for t in gt_boundaries:
            ax.axvline(float(t), color='black', alpha=0.14, lw=0.8)

    fig.tight_layout()
    out_path = FIGURES_DIR / f'timeseries_with_gt_{video.video_id}.png'
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

print('saved per-video timeseries figures to:', FIGURES_DIR)

In [ ]:
# Estadistica global: correlacion y separabilidad

all_curves = []
all_labels = []
curve_labels_ref: List[str] | None = None
for video in videos:
    all_curves.append(video.curve_signals)
    all_labels.append(video.frame_label_ids)
    if curve_labels_ref is None:
        curve_labels_ref = video.curve_labels

X_curves = np.vstack(all_curves).astype(np.float32)
y_labels = np.hstack(all_labels).astype(np.int32)
curve_labels_ref = curve_labels_ref or [f'curve_{i}' for i in range(X_curves.shape[1])]

# Heatmap de correlacion (subconjunto para legibilidad)
max_corr_dim = min(16, X_curves.shape[1])
corr = np.corrcoef(X_curves[:, :max_corr_dim], rowvar=False)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1.0, vmax=1.0)
ax.set_title('Correlation of topological descriptors (subset)')
ax.set_xticks(range(max_corr_dim))
ax.set_yticks(range(max_corr_dim))
ax.set_xticklabels(curve_labels_ref[:max_corr_dim], rotation=90, fontsize=8)
ax.set_yticklabels(curve_labels_ref[:max_corr_dim], fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'descriptor_correlation_heatmap.png', dpi=160)
plt.close(fig)

# Separabilidad simple por Fisher score
fisher_scores = []
classes = np.unique(y_labels)
global_mean = np.mean(X_curves, axis=0)
for j in range(X_curves.shape[1]):
    between = 0.0
    within = 0.0
    for c in classes:
        mask = y_labels == c
        if not np.any(mask):
            continue
        x_c = X_curves[mask, j]
        mean_c = float(np.mean(x_c))
        var_c = float(np.var(x_c))
        n_c = int(mask.sum())
        between += n_c * (mean_c - float(global_mean[j])) ** 2
        within += n_c * var_c
    score = float(between / (within + 1e-8))
    fisher_scores.append(score)
fisher_scores = np.asarray(fisher_scores, dtype=np.float32)

rank_idx = np.argsort(-fisher_scores)[:12]
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.bar(range(rank_idx.shape[0]), fisher_scores[rank_idx], color='tab:orange')
ax.set_xticks(range(rank_idx.shape[0]))
ax.set_xticklabels([curve_labels_ref[i] for i in rank_idx], rotation=45, ha='right')
ax.set_ylabel('Fisher score')
ax.set_title('Descriptor separability by action-unit labels')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'feature_separability_barplot.png', dpi=160)
plt.close(fig)

print('Saved correlation and separability plots.')

In [ ]:
# Boundary detection a priori usando solo descriptores topologicos
threshold_grid = np.linspace(0.70, 0.98, 15)
boundary_rows = []

for q in threshold_grid:
    tp_total = 0
    fp_total = 0
    fn_total = 0
    for video in videos:
        ts = video.timestamps
        gt_times = _segment_boundaries_from_labels(video.frame_label_ids, ts)

        if 'combined_activity_delta' in video.curve_labels:
            d_idx = video.curve_labels.index('combined_activity_delta')
            combined_delta = video.curve_signals[:, d_idx]
        else:
            if 'combined_activity' in video.curve_labels:
                c_idx = video.curve_labels.index('combined_activity')
                raw = video.curve_signals[:, c_idx]
                combined_delta = np.hstack([np.array([0.0], dtype=np.float32), np.abs(np.diff(raw))])
            else:
                combined_delta = np.zeros_like(ts)

        if 'brightness_delta' in video.curve_labels:
            b_idx = video.curve_labels.index('brightness_delta')
            brightness_delta = video.curve_signals[:, b_idx]
        else:
            brightness_delta = np.zeros_like(ts)

        score = _zscore(np.abs(combined_delta)) + 0.35 * _zscore(np.abs(brightness_delta))
        score = _moving_average_1d(score, SMOOTH_WINDOW)

        thr = float(np.quantile(score, q))
        peak_idx = _find_peaks(score, thr)
        pred_times = ts[peak_idx] if peak_idx.size else np.zeros((0,), dtype=np.float32)

        tp, fp, fn, _ = _match_events(pred_times, gt_times, BOUNDARY_TOLERANCE_SEC)
        tp_total += tp
        fp_total += fp
        fn_total += fn

    precision = tp_total / max(1, tp_total + fp_total)
    recall = tp_total / max(1, tp_total + fn_total)
    f1 = 2 * precision * recall / max(1e-8, precision + recall)
    boundary_rows.append({
        'quantile': float(q),
        'tp': int(tp_total),
        'fp': int(fp_total),
        'fn': int(fn_total),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
    })

best = max(boundary_rows, key=lambda x: x['f1']) if boundary_rows else None

fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.plot([r['recall'] for r in boundary_rows], [r['precision'] for r in boundary_rows], marker='o', lw=1.2)
for row in boundary_rows:
    ax.annotate(f"q={row['quantile']:.2f}", (row['recall'], row['precision']), fontsize=7, alpha=0.8)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Boundary detection (TDA-only), tol={BOUNDARY_TOLERANCE_SEC:.2f}s')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'boundary_pr_curve.png', dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8.5, 4.0))
ax.plot([r['quantile'] for r in boundary_rows], [r['f1'] for r in boundary_rows], marker='o', color='tab:green')
ax.set_xlabel('Quantile threshold')
ax.set_ylabel('Boundary F1')
ax.set_title('Boundary F1 vs threshold quantile')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'boundary_f1_vs_threshold.png', dpi=160)
plt.close(fig)

print('Best boundary setting:', best)

In [ ]:
# PCA de descriptores topologicos coloreado por action unit
max_points = 6000
if X_curves.shape[0] > max_points:
    sample_idx = np.random.choice(X_curves.shape[0], size=max_points, replace=False)
else:
    sample_idx = np.arange(X_curves.shape[0])

X_sample = X_curves[sample_idx]
y_sample = y_labels[sample_idx]

pca = PCA(n_components=2, random_state=7)
X_2d = pca.fit_transform(X_sample)

fig, ax = plt.subplots(figsize=(8.5, 6.5))
unique_ids = np.unique(y_sample)
colors = plt.cm.tab20(np.linspace(0, 1, max(1, unique_ids.shape[0])))
for i, label_id in enumerate(unique_ids):
    mask = y_sample == label_id
    label_name = id_to_label.get(int(label_id), f'class_{int(label_id)}')
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], s=8, alpha=0.55, color=colors[i], label=label_name)

ax.set_title('PCA of topological descriptors by action-unit label')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
if unique_ids.shape[0] <= 18:
    ax.legend(loc='best', fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'pca_scatter_labels.png', dpi=170)
plt.close(fig)

explained = pca.explained_variance_ratio_.tolist()
print('PCA explained variance ratio:', explained)

In [ ]:
# Guardar resumen cuantitativo del experimento
summary = {
    'config': {
        'split': SPLIT,
        'max_videos': MAX_VIDEOS,
        'activities': ACTIVITIES,
        'boundary_tolerance_sec': BOUNDARY_TOLERANCE_SEC,
        'smooth_window': SMOOTH_WINDOW,
    },
    'num_videos_loaded': len(videos),
    'videos': [
        {
            'video_id': v.video_id,
            'split': v.split,
            'activity_label': v.activity_label,
            'subject_id': v.subject_id,
            'num_frames': int(v.timestamps.shape[0]),
            'num_gt_segments': int(np.sum(np.diff(v.frame_label_ids) != 0) + 1),
        }
        for v in videos
    ],
    'boundary_grid_results': boundary_rows,
    'best_boundary_result': best,
    'top_descriptors_by_fisher': [
        {
            'descriptor': curve_labels_ref[int(i)],
            'fisher_score': float(fisher_scores[int(i)]),
        }
        for i in rank_idx.tolist()
    ],
}

summary_path = EXPERIMENT_DIR / 'experiment_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('summary saved:', summary_path)

## Interpretacion sugerida

- Si la curva `combined_activity_delta` y el `boundary score` muestran picos cerca de boundaries GT con F1 razonable,
  hay evidencia de que TDA captura transiciones semanticas **antes** de entrenar el modelo temporal.
- Si los mejores descriptores tienen Fisher score alto y el PCA muestra separacion parcial por labels,
  los descriptores topologicos contienen informacion discriminativa para action units.
- Si los resultados son debiles, ajustar:
  - `sample_fps`
  - `smooth_window`
  - combinacion de score para boundaries
  - descriptores usados (incluir/excluir `brightness_delta`, `h1_sum_persistence`, etc.)